# День 7 — Pandas + Matplotlib: мини-EDA

## Цель
Загрузить CSV, почистить, сделать сводки и графики, написать выводы.

## Загрузка и первичный осмотр данных

### Методы первичного осмотра

- `head(n)` — первые n строк (по умолчанию 5) — быстро посмотреть как выглядят данные
- `info()` — типы колонок, количество непустых значений, память
- `describe()` — статистика: count, mean, std, min, max, quartiles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Jupyter из корня репо → data/; из week-2/day-7-pandas-eda → ../../data
DATA_DIR = Path("data")
if not (DATA_DIR / "titanic.csv").exists():
    DATA_DIR = Path("../../data")

df_p = pd.read_csv(DATA_DIR / "perfumes.csv")

print(f"Размер: {df_p.shape}")
print("\nПропуски:")
print(df_p.isnull().sum())


df_p.head()

In [ ]:
# `info()` — типы колонок, количество непустых значений, память
print(df_p.info())

In [ ]:
# `describe()` — статистика: count, mean, std, min, max, quartiles
df_p.describe()

### Чистка данных
#### Пропуски
- `df.isnull().sum()` — количество пропусков в каждой колонке
- `df.isna()` — то же самое (алиас)

#### Заполнить
- `fillna(value)` — заполнить пропуски значением

#### Создать
- `df['new_col'] = ...` — создать новую колонку

#### Удалить
- `df.drop(columns=['col'])` — удалить колонку
- `dropna()` — удалить строки с пропусками

#### Изменить
- `df['col'].astype(int)` — изменить тип данных




In [ ]:
# Пропуски до чистки
print("Пропуски до чистки:")
print(df_p.isnull().sum())

# Числа — заполняем медианой
df_p['price'] = df_p['price'].fillna(df_p['price'].median())
df_p['rating'] = df_p['rating'].fillna(df_p['rating'].median())

# Категория — заполняем самым частым значением
df_p['category'] = df_p['category'].fillna(df_p['category'].mode()[0])

# Новая колонка — ценовой сегмент
df_p['segment'] = df_p['price'].apply(lambda x: 'luxury' if x >= 120 else 'premium' if x >= 80 else 'accessible')

print("\nПосле чистки:")
print(df_p.isnull().sum())
df_p.head()

### groupby + agg

- `df.groupby('col')` — группировать по колонке
- `.mean()` — среднее по группе
- `.sum()` — сумма по группе
- `.count()` — количество по группе
- `.agg({'col1': 'mean', 'col2': 'sum'})` — разные агрегации для разных колонок

### Базовые графики matplotlib

- `plt.hist(data)` — гистограмма (распределение)
- `plt.bar(x, y)` — столбчатая диаграмма
- `plt.plot(x, y)` — линейный график
- `plt.xlabel('...')` — подпись оси X
- `plt.ylabel('...')` — подпись оси Y
- `plt.title('...')` — заголовок
- `plt.show()` — показать график

- `plt.subplots(rows, cols, figsize=(w, h))` — создать несколько графиков в одной фигуре
- `axes[i]` — обращение к конкретному графику
- `plt.tight_layout()` — автоматически выровнять отступы между графиками

- `fig` — вся фигура (холст) на которой рисуются графики
- `axes` — массив графиков внутри фигуры: `axes[0]`, `axes[1]`...
- Аналогия: `fig` — это контейнер, `axes` — элементы внутри

In [ ]:
# Анализ данных — groupby

# 1. Средняя цена по категории
print("1. Средняя цена по категории:")
print(df_p.groupby('category')['price'].mean().round(1))

# 2. Средний рейтинг по сегменту
print("\n2. Средний рейтинг по сегменту:")
print(df_p.groupby('segment')['rating'].mean().round(2))

# 3. Суммарные продажи по категории
print("\n3. Суммарные продажи по категории:")
print(df_p.groupby('category')['sales'].sum())

In [ ]:
# Визуализация

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# График 1 — распределение цен
axes[0].hist(df_p['price'], bins=10, color='orchid')
axes[0].set_title('Распределение цен')
axes[0].set_xlabel('Цена')
axes[0].set_ylabel('Количество')

# График 2 — продажи по категории
sales_by_cat = df_p.groupby('category')['sales'].sum()
axes[1].bar(sales_by_cat.index, sales_by_cat.values, color='lightpink')
axes[1].set_title('Продажи по категории')
axes[1].set_xlabel('Категория')
axes[1].set_ylabel('Продажи')

# График 3 — рейтинг по сегменту
rating_by_seg = df_p.groupby('segment')['rating'].mean()
axes[2].bar(rating_by_seg.index, rating_by_seg.values, color='mediumpurple')
axes[2].set_title('Рейтинг по сегменту')
axes[2].set_xlabel('Сегмент')
axes[2].set_ylabel('Средний рейтинг')

plt.tight_layout()
plt.show()

### Выводы по датасету парфюмов

- Floral парфюмы самые популярные — суммарные продажи 7450
- Woody парфюмы самые дорогие в среднем (130.8)
- Luxury сегмент имеет лучший рейтинг (4.77) — цена коррелирует с качеством
- Oriental парфюмы продаются меньше всего (2600)
- Большинство парфюмов в диапазоне 80-130 (premium сегмент)

## Задания

Датасет: `data/titanic.csv` (891 пассажир)

Вопросы:
1. Какой процент пассажиров выжил?
2. Влияет ли класс билета на выживаемость?
3. Как распределён возраст пассажиров?


In [ ]:
# Загрузка и чистка Titanic
df = pd.read_csv(DATA_DIR / "titanic.csv")

print(f"Размер: {df.shape}")
print(df.isnull().sum())

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(columns=['Cabin'])
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

print("\nПосле чистки:")
print(df.isnull().sum())

In [ ]:
# Ответы на вопросы к данным

# Вопрос 1: процент выживших
print(f"1. Выжило: {df['Survived'].mean() * 100:.1f}%")

# Вопрос 2: выживаемость по классу
print("\n2. Выживаемость по классу:")
print(df.groupby('Pclass')['Survived'].mean().round(2))

# Вопрос 3: распределение возраста
print("\n3. Статистика возраста:")
print(df['Age'].describe().round(1))

In [ ]:
# Агрегации groupby

# 1. Выживаемость по полу
print("1. Выживаемость по полу:")
print(df.groupby('Sex')['Survived'].mean().round(2))

# 2. Средний размер семьи по классу
print("\n2. Средний размер семьи по классу:")
print(df.groupby('Pclass')['FamilySize'].mean().round(1))

# 3. Средняя цена билета по классу
print("\n3. Средняя цена билета по классу:")
print(df.groupby('Pclass')['Fare'].mean().round(1))

In [ ]:
# Графики
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# График 1 — распределение возраста
axes[0].hist(df['Age'], bins=30, color='steelblue')
axes[0].set_title('Распределение возраста')
axes[0].set_xlabel('Возраст')
axes[0].set_ylabel('Количество')

# График 2 — выживаемость по классу
survival_by_class = df.groupby('Pclass')['Survived'].mean()
axes[1].bar(survival_by_class.index, survival_by_class.values, color='coral')
axes[1].set_title('Выживаемость по классу')
axes[1].set_xlabel('Класс')
axes[1].set_ylabel('Доля выживших')

# График 3 — выживаемость по полу
survival_by_sex = df.groupby('Sex')['Survived'].mean()
axes[2].bar(survival_by_sex.index, survival_by_sex.values, color='mediumseagreen')
axes[2].set_title('Выживаемость по полу')
axes[2].set_xlabel('Пол')
axes[2].set_ylabel('Доля выживших')

plt.tight_layout()
plt.show()

### Выводы

- Датасет: 891 пассажир, после чистки пропусков в Age/Embarked нет (Cabin удалён)
- Выжило около 38% пассажиров
- Класс сильно влияет: выживаемость растёт от 3-го к 1-му классу
- Женщины выживали чаще мужчин
- Возраст: большинство пассажиров 20–40 лет
- Средняя цена билета заметно выше в 1-м классе
- Размер семьи примерно одинаков по классам (~1.8–2.0)

### Гипотезы для дальнейшего анализа
- Семьи среднего размера выживали чаще одиночек?
- Цена билета коррелирует с выживаемостью независимо от класса?